# 통합 실행과 운영 한계

통합 실행의 사용량을 읽고 비용·권한·정지 조건을 운영 기록으로 남깁니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

한 가지 동작을 실행하고 출력으로 확인합니다.

In [3]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemPermission

# 이 노트북만의 새 작업 폴더와 차단 규칙을 준비합니다.
workspace = Path('labs/part6/ch27-deepagents/workspace-08').resolve()
workspace.mkdir(parents=True, exist_ok=True)
backend = FilesystemBackend(root_dir=workspace)
deny_blocked = FilesystemPermission(operations=['write'], paths=['/blocked.txt'], mode='deny')
agent = create_deep_agent(model=init_chat_model('openai:gpt-5.4-mini'), backend=backend, permissions=[deny_blocked])

## 단계 2

한 가지 동작을 실행하고 출력으로 확인합니다.

In [4]:
# 허용 경로에는 보고서를 쓰고 차단 경로에는 쓰기를 시도합니다.
prompt = '''write_file로 /run/report.md에 "status: complete"를 쓰세요.
그 다음 /blocked.txt에도 같은 내용을 쓰려고 시도하고, 두 결과를 설명한 뒤 종료하세요.'''
result = agent.invoke({'messages':[{'role':'user','content':prompt}]})
tool_messages = [m for m in result['messages'] if type(m).__name__ == 'ToolMessage']
for message in tool_messages: print(message.name, str(message.content)[:300])

write_file Updated file /run/report.md
write_file Error: permission denied for write on /blocked.txt


## 단계 3

한 가지 동작을 실행하고 출력으로 확인합니다.

In [5]:
# 종료 조건과 권한 거부를 실제 파일과 도구 메시지로 확인합니다.
report = backend.read('/run/report.md')
denied = [m for m in tool_messages if 'permission' in str(m.content).lower() or 'denied' in str(m.content).lower()]
print('보고서:', report.file_data['content'])
print('최종 답변:', result['messages'][-1].text)
assert report.file_data['content'].strip() == 'status: complete'
assert denied
assert result['messages'][-1].text

보고서: status: complete
최종 답변: `/run/report.md`에 `status: complete`를 성공적으로 썼습니다.

그 다음 `/blocked.txt`에도 같은 내용을 쓰려고 시도했지만, 권한 거부(`permission denied for write on /blocked.txt`)로 실패했습니다.

요약:
- `/run/report.md`: 성공
- `/blocked.txt`: 실패, 권한 문제
